# 📊 Modul Simulasi Diagram Bode
### Sistem Kontrol — Teknik Elektro

---

**Tujuan Pembelajaran:**
1. Memahami konsep diagram Bode sebagai representasi respons frekuensi
2. Menggambar plot magnitude (dB) dan fase (°) dari fungsi alih
3. Menggunakan aproksimasi asimtot untuk sketsa cepat
4. Mengidentifikasi gain margin dan phase margin
5. Menganalisis kestabilan sistem dari diagram Bode

**Prasyarat:** Fungsi alih, transformasi Laplace, bilangan kompleks

---

## ⚙️ Instalasi Library (jalankan sekali di Colab)

In [ ]:
# Jalankan sel ini jika menggunakan Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install control -q
    print('✅ Library control berhasil diinstall')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import control
from ipywidgets import interact, FloatSlider, FloatLogSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'axes.grid.which': 'both',
    'grid.alpha': 0.3,
    'font.size': 11
})
print('✅ Semua library berhasil diimport')

---
## 1. Konsep Dasar Diagram Bode

### 1.1 Respons Frekuensi

Respons frekuensi suatu sistem LTI (Linear Time Invariant) diperoleh dengan mengganti $s = j\omega$ pada fungsi alih $H(s)$:

$$H(j\omega) = |H(j\omega)| \cdot e^{j\angle H(j\omega)}$$

**Diagram Bode** terdiri dari dua plot terhadap frekuensi $\omega$ (skala logaritmik):

| Plot | Sumbu Y | Rumus |
|------|---------|-------|
| **Magnitude** | dB | $|H(j\omega)|_{dB} = 20\log_{10}|H(j\omega)|$ |
| **Fase** | Derajat | $\angle H(j\omega) = \arctan\left(\frac{\text{Im}[H(j\omega)]}{\text{Re}[H(j\omega)]}\right)$ |

### 1.2 Keunggulan Skala Logaritmik

Jika $H(s) = H_1(s) \cdot H_2(s)$, maka:
$$|H|_{dB} = |H_1|_{dB} + |H_2|_{dB}$$
$$\angle H = \angle H_1 + \angle H_2$$

> 💡 **Kunci:** Dengan skala dB, perkalian fungsi alih menjadi **penjumlahan** di diagram Bode!

---
## 2. Sistem Orde Pertama

### Fungsi Alih

$$H(s) = \frac{K}{\tau s + 1} = \frac{K/\tau}{s + 1/\tau}$$

Dengan $s = j\omega$:
$$H(j\omega) = \frac{K}{j\omega\tau + 1}$$

**Magnitude:**
$$|H(j\omega)|_{dB} = 20\log_{10}K - 20\log_{10}\sqrt{1 + (\omega\tau)^2}$$

**Fase:**
$$\angle H(j\omega) = -\arctan(\omega\tau)$$

**Frekuensi Corner:** $\omega_c = \dfrac{1}{\tau}$ rad/s

In [ ]:
def plot_bode_orde1(K=1.0, tau=1.0):
    """Plot diagram Bode sistem orde pertama H(s) = K/(tau*s + 1)"""
    omega = np.logspace(-2, 3, 1000)  # 0.01 hingga 1000 rad/s
    
    # Hitung H(jw)
    H = K / (1j * omega * tau + 1)
    mag_dB = 20 * np.log10(np.abs(H))
    phase_deg = np.degrees(np.angle(H))
    
    # Aproksimasi asimtot
    wc = 1 / tau
    mag_asymp = np.where(
        omega < wc,
        20 * np.log10(K) * np.ones_like(omega),
        20 * np.log10(K) - 20 * np.log10(omega * tau)
    )
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))
    fig.suptitle(f'Diagram Bode — Sistem Orde 1: H(s) = {K:.1f}/({tau:.2f}s + 1)', 
                 fontsize=13, fontweight='bold')
    
    # Plot Magnitude
    ax1.semilogx(omega, mag_dB, 'b-', linewidth=2.5, label='Kurva eksak')
    ax1.semilogx(omega, mag_asymp, 'r--', linewidth=1.5, label='Aproksimasi asimtot')
    ax1.axvline(x=wc, color='g', linestyle=':', linewidth=1.5, label=f'ωc = {wc:.2f} rad/s')
    ax1.axhline(y=20*np.log10(K)-3, color='orange', linestyle=':', alpha=0.7, label=f'−3 dB = {20*np.log10(K)-3:.1f} dB')
    ax1.set_ylabel('Magnitude (dB)', fontsize=11)
    ax1.legend(loc='lower left', fontsize=9)
    ax1.set_title(f'Plot Magnitude  |  K = {K:.2f}, τ = {tau:.3f} s', fontsize=10)
    
    # Anotasi nilai di wc
    mag_at_wc = 20 * np.log10(K / np.sqrt(2))
    ax1.annotate(f'({wc:.2f}, {mag_at_wc:.1f} dB)', 
                xy=(wc, mag_at_wc), xytext=(wc*3, mag_at_wc+5),
                arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=9)
    
    # Plot Fase
    ax2.semilogx(omega, phase_deg, 'b-', linewidth=2.5, label='Kurva eksak')
    ax2.axvline(x=wc, color='g', linestyle=':', linewidth=1.5, label=f'ωc = {wc:.2f} rad/s')
    ax2.axhline(y=-45, color='orange', linestyle=':', alpha=0.7, label='-45° pada ωc')
    ax2.set_xlabel('Frekuensi ω (rad/s)', fontsize=11)
    ax2.set_ylabel('Fase (derajat)', fontsize=11)
    ax2.set_ylim([-100, 10])
    ax2.legend(loc='lower left', fontsize=9)
    ax2.set_title('Plot Fase', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📌 Ringkasan Parameter:")
    print(f"   Gain DC (ω→0)    : {20*np.log10(K):.2f} dB")
    print(f"   Frekuensi corner : ωc = 1/τ = {wc:.4f} rad/s = {wc/(2*np.pi):.4f} Hz")
    print(f"   Magnitude di ωc  : {mag_at_wc:.2f} dB (= gain DC − 3 dB)")
    print(f"   Fase di ωc       : −45°")
    print(f"   Slope HF         : −20 dB/dekade")

# Demo langsung
plot_bode_orde1(K=2.0, tau=0.5)

### 🎛️ Eksplorasi Interaktif — Sistem Orde 1
Geser slider untuk melihat pengaruh parameter terhadap diagram Bode:

In [ ]:
interact(
    plot_bode_orde1,
    K=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='K (Gain)'),
    tau=FloatLogSlider(value=1.0, min=-2, max=2, step=0.1, description='τ (s)', base=10)
);

---
## 3. Sistem Orde Kedua

### Fungsi Alih Standar

$$H(s) = \frac{K\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

di mana:
- $\omega_n$ = frekuensi natural (rad/s)
- $\zeta$ = rasio redaman (damping ratio)
- $K$ = gain DC

| Kondisi | Nilai ζ | Karakteristik |
|---------|---------|---------------|
| Underdamped | $0 < \zeta < 1$ | Ada puncak resonansi |
| Critically damped | $\zeta = 1$ | Tanpa overshoot |
| Overdamped | $\zeta > 1$ | Respons lambat |

**Puncak resonansi (untuk $\zeta < 1/\sqrt{2} \approx 0.707$):**
$$M_p = \frac{1}{2\zeta\sqrt{1-\zeta^2}} \quad \text{pada} \quad \omega_r = \omega_n\sqrt{1-2\zeta^2}$$

In [ ]:
def plot_bode_orde2(wn=1.0, zeta=0.5, K=1.0):
    """Plot Bode sistem orde 2: H(s) = K*wn^2 / (s^2 + 2*zeta*wn*s + wn^2)"""
    omega = np.logspace(-1, 2, 2000) * wn
    omega = np.logspace(np.log10(wn/20), np.log10(wn*50), 2000)
    
    # Evaluasi H(jw)
    s = 1j * omega
    H = K * wn**2 / (s**2 + 2*zeta*wn*s + wn**2)
    mag_dB = 20 * np.log10(np.abs(H))
    phase_deg = np.degrees(np.angle(H))
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))
    fig.suptitle(f'Diagram Bode — Sistem Orde 2 | ωₙ={wn:.2f}, ζ={zeta:.3f}, K={K:.1f}',
                 fontsize=13, fontweight='bold')
    
    ax1.semilogx(omega, mag_dB, 'b-', linewidth=2.5)
    ax1.axvline(x=wn, color='r', linestyle='--', linewidth=1.5, label=f'ωₙ = {wn:.2f} rad/s')
    
    # Tandai puncak resonansi jika ada
    if zeta < 1/np.sqrt(2):
        wr = wn * np.sqrt(1 - 2*zeta**2)
        Mp_dB = 20 * np.log10(K / (2*zeta*np.sqrt(1 - zeta**2)))
        ax1.axvline(x=wr, color='purple', linestyle=':', linewidth=1.5, 
                    label=f'ωr = {wr:.2f} rad/s')
        ax1.plot(wr, Mp_dB, 'r*', markersize=12, label=f'Puncak = {Mp_dB:.1f} dB')
    
    ax1.set_ylabel('Magnitude (dB)')
    ax1.legend(fontsize=9)
    
    ax2.semilogx(omega, phase_deg, 'b-', linewidth=2.5)
    ax2.axvline(x=wn, color='r', linestyle='--', linewidth=1.5, label=f'ωₙ = {wn:.2f} rad/s')
    ax2.axhline(y=-90, color='gray', linestyle=':', alpha=0.7, label='-90° pada ωₙ')
    ax2.set_xlabel('Frekuensi ω (rad/s)')
    ax2.set_ylabel('Fase (derajat)')
    ax2.set_ylim([-200, 10])
    ax2.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Perbandingan berbagai nilai zeta
omega = np.logspace(-1, 1, 1000)  # sekitar wn=1
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))
fig.suptitle('Pengaruh Rasio Redaman ζ pada Diagram Bode (ωₙ = 1 rad/s)', 
             fontsize=13, fontweight='bold')

zeta_list = [0.1, 0.3, 0.5, 0.707, 1.0, 2.0]
colors = ['#e41a1c','#ff7f00','#4daf4a','#984ea3','#377eb8','#a65628']

for zeta, color in zip(zeta_list, colors):
    s = 1j * omega
    H = 1 / (s**2 + 2*zeta*s + 1)
    mag = 20 * np.log10(np.abs(H))
    phase = np.degrees(np.angle(H))
    label = f'ζ = {zeta}'
    ax1.semilogx(omega, mag, color=color, linewidth=2, label=label)
    ax2.semilogx(omega, phase, color=color, linewidth=2, label=label)

ax1.axvline(x=1, color='k', linestyle=':', alpha=0.5)
ax1.set_ylabel('Magnitude (dB)')
ax1.legend(fontsize=9, ncol=2)
ax1.set_ylim([-60, 20])

ax2.axvline(x=1, color='k', linestyle=':', alpha=0.5)
ax2.axhline(y=-90, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Frekuensi ω/ωₙ')
ax2.set_ylabel('Fase (derajat)')
ax2.set_ylim([-200, 10])
ax2.legend(fontsize=9, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
interact(
    plot_bode_orde2,
    wn=FloatLogSlider(value=1.0, min=-1, max=2, step=0.1, description='ωₙ (rad/s)', base=10),
    zeta=FloatSlider(value=0.5, min=0.05, max=3.0, step=0.05, description='ζ (damping)'),
    K=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='K (Gain)')
);

---
## 4. Aproksimasi Asimtot

Asimtot Bode dibuat dari aturan sederhana untuk tiap **faktor primitif**:

| Faktor | Slope Magnitude | Kontribusi Fase |
|--------|----------------|------------------|
| Gain $K$ | 0 dB/dec | 0° (atau ±180°) |
| Integrator $1/s$ | −20 dB/dec | −90° |
| Differentiator $s$ | +20 dB/dec | +90° |
| Pole $(\tau s+1)^{-1}$ | −20 dB/dec setelah $\omega_c$ | −45° pada $\omega_c$ |
| Zero $(\tau s+1)$ | +20 dB/dec setelah $\omega_c$ | +45° pada $\omega_c$ |
| Orde-2 underdamped | −40 dB/dec setelah $\omega_n$ | −90° pada $\omega_n$ |

> 💡 **Tip:** Untuk fase, asimtot berubah 1 dekade sebelum dan sesudah frekuensi corner.

In [ ]:
# Demonstrasi superposisi: H(s) = 10(s+1) / [s(s+10)]
# = 10*(s+1) / [s*(s+10)]
# Faktor: gain 10/10=1, zero (s+1), integrator 1/s, pole 1/(s/10+1)

omega = np.logspace(-2, 3, 2000)
s = 1j * omega

# Fungsi alih lengkap: H(s) = 10(s+1) / [s(s+10)]
H_total = 10 * (s + 1) / (s * (s + 10))

# Komponen-komponen
H_gain = 10/10 * np.ones_like(omega, dtype=complex)       # gain = 1 (0 dB)
H_zero = (1j*omega + 1)                                    # zero di s=-1
H_integrator = 1 / (1j*omega)                              # integrator
H_pole = 1 / (1j*omega/10 + 1)                            # pole di s=-10

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('Superposisi Diagram Bode\nH(s) = 10(s+1) / [s(s+10)]', 
             fontsize=13, fontweight='bold')

# Magnitude
ax1 = axes[0]
ax1.semilogx(omega, 20*np.log10(np.abs(H_total)), 'k-', linewidth=3, label='H total (eksak)', zorder=5)
ax1.semilogx(omega, 20*np.log10(np.abs(H_zero)), 'g--', linewidth=1.5, alpha=0.8, label='Zero (s+1)')
ax1.semilogx(omega, 20*np.log10(np.abs(H_integrator)), 'r--', linewidth=1.5, alpha=0.8, label='Integrator 1/s')
ax1.semilogx(omega, 20*np.log10(np.abs(H_pole)), 'b--', linewidth=1.5, alpha=0.8, label='Pole 1/(s/10+1)')
ax1.axvline(x=1, color='g', linestyle=':', alpha=0.5)
ax1.axvline(x=10, color='b', linestyle=':', alpha=0.5)
ax1.text(1, 25, 'ωc=1', color='g', fontsize=9)
ax1.text(10, 25, 'ωc=10', color='b', fontsize=9)
ax1.set_ylabel('Magnitude (dB)')
ax1.legend(loc='lower left', fontsize=9)

# Fase
ax2 = axes[1]
# Unwrap phase untuk kontinuitas
ph_total = np.unwrap(np.angle(H_total))
ax2.semilogx(omega, np.degrees(ph_total), 'k-', linewidth=3, label='H total (eksak)', zorder=5)
ax2.semilogx(omega, np.degrees(np.angle(H_zero)), 'g--', linewidth=1.5, alpha=0.8, label='Zero (s+1)')
ax2.semilogx(omega, np.degrees(np.angle(H_integrator)), 'r--', linewidth=1.5, alpha=0.8, label='Integrator 1/s')
ax2.semilogx(omega, np.degrees(np.angle(H_pole)), 'b--', linewidth=1.5, alpha=0.8, label='Pole 1/(s/10+1)')
ax2.set_xlabel('Frekuensi ω (rad/s)')
ax2.set_ylabel('Fase (derajat)')
ax2.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()
print('\n📌 Perhatikan bagaimana kurva total = PENJUMLAHAN semua komponen!')

---
## 5. Gain Margin dan Phase Margin

### Definisi

Untuk **sistem loop tertutup** dengan gain $K$:

$$\text{Gain Margin (GM)} = -|G(j\omega_{pc})|_{dB}$$
di mana $\omega_{pc}$ = **frekuensi phase crossover** ($\angle G = -180°$)

$$\text{Phase Margin (PM)} = 180° + \angle G(j\omega_{gc})$$
di mana $\omega_{gc}$ = **frekuensi gain crossover** ($|G| = 0$ dB)

### Kriteria Kestabilan

| Kondisi | Interpretasi |
|---------|-------------|
| GM > 0 dB **dan** PM > 0° | Sistem **stabil** |
| GM = 0 dB **atau** PM = 0° | Sistem **pada batas kestabilan** |
| GM < 0 dB **atau** PM < 0° | Sistem **tidak stabil** |

In [ ]:
def analisis_margin(num, den, title='Sistem Loop Terbuka'):
    """Analisis gain margin dan phase margin menggunakan library control"""
    G = control.tf(num, den)
    
    # Hitung margin
    gm, pm, wpc, wgc = control.margin(G)
    gm_dB = 20 * np.log10(gm) if gm > 0 else float('inf')
    
    # Plot Bode
    omega = np.logspace(-1, 2, 2000)
    H = np.polyval(num, 1j*omega) / np.polyval(den, 1j*omega)
    mag_dB = 20 * np.log10(np.abs(H))
    phase_deg = np.degrees(np.unwrap(np.angle(H)))
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    fig.suptitle(f'Diagram Bode & Analisis Margin\n{title}', fontsize=12, fontweight='bold')
    
    # Magnitude plot
    ax1.semilogx(omega, mag_dB, 'b-', linewidth=2.5)
    ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.8)
    if wgc and not np.isnan(wgc):
        mag_at_wgc = 0  # by definition
        ax1.axvline(x=wgc, color='r', linestyle='--', linewidth=1.5, label=f'ωgc = {wgc:.3f} rad/s')
        ax1.plot(wgc, 0, 'ro', markersize=10)
    if wpc and not np.isnan(wpc):
        mag_at_wpc = 20 * np.log10(np.abs(np.polyval(num, 1j*wpc) / np.polyval(den, 1j*wpc)))
        ax1.axvline(x=wpc, color='g', linestyle='--', linewidth=1.5, label=f'ωpc = {wpc:.3f} rad/s')
        ax1.annotate('', xy=(wpc, 0), xytext=(wpc, mag_at_wpc),
                    arrowprops=dict(arrowstyle='<->', color='green', lw=2))
        ax1.text(wpc*1.1, mag_at_wpc/2, f'GM = {gm_dB:.1f} dB', color='green', fontsize=10)
    ax1.set_ylabel('Magnitude (dB)')
    ax1.legend(fontsize=9)
    
    # Phase plot
    ax2.semilogx(omega, phase_deg, 'b-', linewidth=2.5)
    ax2.axhline(y=-180, color='k', linestyle='-', linewidth=0.8)
    if wgc and not np.isnan(wgc):
        ph_at_wgc = np.degrees(np.angle(np.polyval(num, 1j*wgc) / np.polyval(den, 1j*wgc)))
        ax2.axvline(x=wgc, color='r', linestyle='--', linewidth=1.5)
        ax2.annotate('', xy=(wgc, -180), xytext=(wgc, ph_at_wgc),
                    arrowprops=dict(arrowstyle='<->', color='red', lw=2))
        ax2.text(wgc*1.1, (-180+ph_at_wgc)/2, f'PM = {pm:.1f}°', color='red', fontsize=10)
        ax2.plot(wgc, ph_at_wgc, 'ro', markersize=10, label=f'PM = {pm:.1f}°')
    if wpc and not np.isnan(wpc):
        ax2.axvline(x=wpc, color='g', linestyle='--', linewidth=1.5)
        ax2.plot(wpc, -180, 'g^', markersize=10, label=f'wpc = {wpc:.3f}')
    ax2.set_xlabel('Frekuensi ω (rad/s)')
    ax2.set_ylabel('Fase (derajat)')
    ax2.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Ringkasan
    print(f"\n{'='*50}")
    print(f"  HASIL ANALISIS MARGIN KESTABILAN")
    print(f"{'='*50}")
    print(f"  Gain Margin  (GM)  : {gm_dB:.2f} dB  @ ω = {wpc:.4f} rad/s")
    print(f"  Phase Margin (PM)  : {pm:.2f}°   @ ω = {wgc:.4f} rad/s")
    print(f"{'='*50}")
    if gm_dB > 0 and pm > 0:
        print(f"  ✅ Sistem STABIL (GM > 0 dB dan PM > 0°)")
    else:
        print(f"  ❌ Sistem TIDAK STABIL")
    print(f"{'='*50}\n")

# Contoh: G(s) = 10 / [s(s+1)(s+5)]
analisis_margin(
    num=[10],
    den=np.polymul([1, 1, 0], [1, 5]),  # s*(s+1) * (s+5)
    title='G(s) = 10 / [s(s+1)(s+5)]'
)

---
## 6. Penggunaan Library `control`

Library Python `control` menyediakan fungsi lengkap untuk analisis sistem kontrol.

In [ ]:
# ======================================================
# Cara membuat Transfer Function dengan library control
# ======================================================

# Metode 1: dari koefisien polinomial
# G(s) = (s + 2) / (s^2 + 3s + 2)
G1 = control.tf([1, 2], [1, 3, 2])
print('G1(s) =', G1)

# Metode 2: dari pole-zero-gain
# G(s) = 5*(s+1) / [(s+2)(s+3)]
G2 = control.zpk([-1], [-2, -3], 5)
print('G2(s) =', G2)

# Operasi seri dan paralel
G_seri = control.series(G1, G2)
G_paralel = control.parallel(G1, G2)
G_loop_tertutup = control.feedback(G1, 1)  # unity feedback

print("\nG seri =", G_seri)
print("\nG loop tertutup unity =", G_loop_tertutup)

In [ ]:
# Plot Bode dengan library control
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Contoh sistem
systems = {
    'G1: (s+2)/(s²+3s+2)': control.tf([1, 2], [1, 3, 2]),
    'G2: 5(s+1)/[(s+2)(s+3)]': control.zpk([-1], [-2, -3], 5),
}

for label, G in systems.items():
    omega = np.logspace(-2, 2, 1000)
    mag, phase, _ = control.bode(G, omega, plot=False)
    mag_dB = 20 * np.log10(mag)
    phase_deg = np.degrees(phase)
    axes[0].semilogx(omega, mag_dB, linewidth=2, label=label)
    axes[1].semilogx(omega, phase_deg, linewidth=2, label=label)

axes[0].set_ylabel('Magnitude (dB)')
axes[0].legend(fontsize=9)
axes[0].set_title('Perbandingan Diagram Bode')
axes[1].set_xlabel('Frekuensi ω (rad/s)')
axes[1].set_ylabel('Fase (derajat)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Ringkasan & Aturan Praktis

### Checklist Menggambar Diagram Bode Manual

1. **Faktorkan** $H(s)$ ke dalam bentuk baku (gain, pole, zero, integrator)
2. **Hitung gain DC** = $20\log_{10}|H(0)|$ dB
3. **Gambar asimtot** untuk tiap faktor (tandai frekuensi corner)
4. **Jumlahkan** semua kontribusi dB dan derajat
5. **Koreksi** ±3 dB di setiap frekuensi corner (magnitude)
6. **Verifikasi** dengan simulasi Python

### Tabel Aturan Slope

| Tambahan | Slope Mag | Fase Akhir |
|----------|-----------|------------|
| 1 pole real | −20 dB/dec | −90° |
| 2 pole (orde 2) | −40 dB/dec | −180° |
| 1 zero real | +20 dB/dec | +90° |
| Integrator $1/s$ | −20 dB/dec | −90° konstan |
| Differentiator $s$ | +20 dB/dec | +90° konstan |

---

## 🏁 Selesai!

Lanjutkan ke file **`tugas_bode_diagram.ipynb`** untuk mengerjakan soal-soal.

Gunakan juga **`bode_interactive.html`** untuk eksplorasi visual secara cepat di browser.